# Data Exploration & Analysis

This notebook performs exploratory data analysis on the movie review sentiment dataset.

## Objectives
- Load and inspect the dataset
- Analyze sentiment distribution
- Examine text characteristics
- Identify data quality issues
- Create visualizations

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import yaml

# Set visualization styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Configuration

In [ ]:
# TODO: Load params.yaml
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print("Configuration loaded:")
print(yaml.dump(params, default_flow_style=False))

## 3. Load Dataset

In [ ]:
# TODO: Load the dataset from CSV
data_path = params['data']['dataset_path']

# Check if file exists
if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"Dataset loaded from {data_path}")
else:
    print(f"Dataset not found at {data_path}")
    print("TODO: Download dataset using src/download_data.py")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'text': ['This movie was great', 'I hated this film', 'It was okay'],
        'label': [1, 0, 1]
    })

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 4. Basic Statistics

In [ ]:
# TODO: Display basic statistics
print("\nDataset Info:")
print(df.info())

print("\nFirst few rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

## 5. Sentiment Distribution

In [ ]:
# TODO: Analyze sentiment distribution
sentiment_counts = df['label'].value_counts()
sentiment_pct = df['label'].value_counts(normalize=True) * 100

print("\nSentiment Distribution:")
for label, count in sentiment_counts.items():
    pct = sentiment_pct[label]
    sentiment_name = 'Positive' if label == 1 else 'Negative'
    print(f"{sentiment_name}: {count} ({pct:.2f}%)")

In [ ]:
# TODO: Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sentiment_labels = ['Negative', 'Positive']
colors = ['#ff6b6b', '#51cf66']
axes[0].bar(sentiment_labels, sentiment_counts.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Sentiment Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(sentiment_counts.values, labels=sentiment_labels, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Sentiment Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/sentiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to results/sentiment_distribution.png")

## 6. Text Analysis

In [ ]:
# TODO: Analyze text characteristics
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("\nText Length Statistics:")
print(df['text_length'].describe())

print("\nWord Count Statistics:")
print(df['word_count'].describe())

In [ ]:
# TODO: Compare text lengths by sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count by sentiment
for label in [0, 1]:
    data = df[df['label'] == label]['word_count']
    sentiment = 'Positive' if label == 1 else 'Negative'
    axes[0].hist(data, alpha=0.6, label=sentiment, bins=30)

axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution by Sentiment', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
df.boxplot(column='word_count', by='label', ax=axes[1])
axes[1].set_xticklabels(['Negative', 'Positive'])
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')
axes[1].set_title('Word Count by Sentiment', fontsize=14, fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../results/text_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Data Quality Issues

In [ ]:
# TODO: Identify data quality issues
print("\nData Quality Assessment:")
print(f"Total samples: {len(df)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Empty texts: {(df['text'].str.len() == 0).sum()}")
print(f"Single-word texts: {(df['word_count'] == 1).sum()}")

# Check class balance
class_ratio = sentiment_counts[1] / sentiment_counts[0]
print(f"\nClass balance (Positive/Negative ratio): {class_ratio:.2f}")

if class_ratio < 0.5 or class_ratio > 2:
    print("⚠️  WARNING: Imbalanced dataset detected. Consider using weighted loss or SMOTE.")
else:
    print("✓ Dataset is reasonably balanced")

## 8. Sample Reviews

In [ ]:
# TODO: Show sample positive reviews
print("Sample Positive Reviews:")
print("="*80)
positive_samples = df[df['label'] == 1]['text'].sample(min(3, len(df[df['label'] == 1])))
for i, text in enumerate(positive_samples, 1):
    print(f"\n{i}. {text[:200]}..." if len(text) > 200 else f"\n{i}. {text}")

print("\n" + "="*80)
print("Sample Negative Reviews:")
print("="*80)
negative_samples = df[df['label'] == 0]['text'].sample(min(3, len(df[df['label'] == 0])))
for i, text in enumerate(negative_samples, 1):
    print(f"\n{i}. {text[:200]}..." if len(text) > 200 else f"\n{i}. {text}")

## 9. Key Findings

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS FROM EDA")
print("="*80)

findings = f"""
1. DATASET SIZE
   - Total reviews: {len(df):,}
   - Positive: {sentiment_counts[1]:,} ({sentiment_pct[1]:.1f}%)
   - Negative: {sentiment_counts[0]:,} ({sentiment_pct[0]:.1f}%)

2. TEXT CHARACTERISTICS
   - Average word count: {df['word_count'].mean():.1f}
   - Average text length: {df['text_length'].mean():.0f} characters
   - Max word count: {df['word_count'].max()}
   - Min word count: {df['word_count'].min()}

3. DATA QUALITY
   - Missing values: {df.isnull().sum().sum()}
   - Duplicates: {df.duplicated().sum()}
   - Empty texts: {(df['text'].str.len() == 0).sum()}

4. RECOMMENDATIONS
   - Use max_length={params['train']['max_length']} for tokenization
   - Consider weighted loss due to potential class imbalance
   - Implement data augmentation if needed
   - Use stratified train/test split
"""

print(findings)